# Part 1 Raw REFIT House 1 cleaning

This notebook inspects the raw REFIT data for House 1 and creates a reproducible cleaning pipeline.


In [1]:
# Import standard libraries and project path setup.

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Make the repository's src/ directory importable from this notebook.
PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cleaning import clean_and_resample, inspect_raw, plot_before_after, write_json
from data_loading import find_house_files, load_raw_house

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "REFIT_RAW_081116"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
PLOTS_DIR = PROJECT_ROOT / "plots"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")

Project root: D:\FLOCK_ENERGY_ASSIGNMENT\refit-appliance-disaggregation
Raw data directory: D:\FLOCK_ENERGY_ASSIGNMENT\refit-appliance-disaggregation\data\raw\REFIT_RAW_081116


In [2]:
# Load and inspect raw data for house 1.
house_files = find_house_files(RAW_DIR, house_number=1)
print("House 1 files:")
for file in house_files:
    print(f"- {file.name}")

raw_house1 = load_raw_house(RAW_DIR, house_number=1)
print(f"Shape: {raw_house1.shape}")
display(raw_house1.head())

House 1 files:
- RAW_House1_Part1.csv
- RAW_House1_Part2.csv
Shape: (8518763, 12)


,Time,Unix,Aggregate,Appliance1,Appliance2,Appliance3,Appliance4,Appliance5,Appliance6,Appliance7,Appliance8,Appliance9
0,2013-10-09 14:06:17,1381327577,523.0,74.0,0.0,69.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2013-10-09 14:06:31,1381327591,526.0,75.0,0.0,69.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2013-10-09 14:06:46,1381327606,540.0,74.0,0.0,68.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2013-10-09 14:07:01,1381327621,532.0,74.0,0.0,68.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2013-10-09 14:07:15,1381327634,540.0,74.0,0.0,69.0,0.0,0.0,0.0,0.0,0.0,1.0


In [3]:
# Inspect the raw data for quality issues.
raw_quality = inspect_raw(raw_house1)

print(f"Rows: {raw_quality['rows']:,}")
print(f"Invalid timestamps: {raw_quality['invalid_timestamps']:,}")
print(f"Duplicate timestamp rows: {raw_quality['duplicate_timestamps']:,}")
print(f"Timestamp start: {raw_quality['timestamp_start']}")
print(f"Timestamp end: {raw_quality['timestamp_end']}")
print("\nTimestamp interval summary (seconds):")
display(pd.Series(raw_quality["interval_seconds"]))

print("Missing values by column:")
display(pd.Series(raw_quality["missing_values"]).sort_values(ascending=False))

print("Negative values by column:")
display(pd.Series(raw_quality["negative_values"]))

print("Values above the 10,000 W inspection limit:")
display(pd.Series(raw_quality["values_above_limit_watts"]))

Rows: 8,518,763
Invalid timestamps: 0
Duplicate timestamp rows: 1,412,513
Timestamp start: 2013-10-09T14:06:17
Timestamp end: 2015-07-10T11:56:32

Timestamp interval summary (seconds):


count              8518762.0
median                   6.0
minimum                  0.0
maximum            3594413.0
irregular_count    7593992.0
dtype: float64

Missing values by column:


Appliance9    1911811
Appliance4     510963
Appliance5     497223
Appliance2     459468
Appliance1     437890
Appliance8     390705
Appliance6     342337
Appliance3     311961
Appliance7     232752
Aggregate      188973
Time                0
Unix                0
dtype: int64

Negative values by column:


Aggregate     0
Appliance1    0
Appliance2    0
Appliance3    0
Appliance4    0
Appliance5    0
Appliance6    0
Appliance7    0
Appliance8    0
Appliance9    0
dtype: int64

Values above the 10,000 W inspection limit:


Aggregate     21097
Appliance1      197
Appliance2       22
Appliance3       47
Appliance4      287
Appliance5       37
Appliance6       19
Appliance7       59
Appliance8       55
Appliance9       18
dtype: int64

In [4]:
# Clean and resample the raw data for house 1.
cleaned_house1, cleaning_report = clean_and_resample(
    raw_house1,
    frequency="1min",
    max_gap_minutes=5,
    max_power_watts=10_000,
)

print("Cleaning report:")
display(pd.Series(cleaning_report))
display(cleaned_house1.head())

Cleaning report:


input_rows                                                               8518763
invalid_timestamps_removed                                                     0
duplicate_rows_removed                                                    761551
resampled_rows                                                            920031
short_gap_rows_filled                                                       8352
long_gap_rows_flagged                                                     214751
remaining_missing_by_column    {'Aggregate': 114630, 'Appliance1': 115521, 'A...
frequency                                                                   1min
short_gap_limit_minutes                                                        5
impossible_value_rule          Values below 0 W or above 10000 W were set to ...
dtype: object

,Aggregate,Appliance1,Appliance2,Appliance3,Appliance4,Appliance5,Appliance6,Appliance7,Appliance8,Appliance9,short_gap_filled,long_gap_missing,observed
Time,,,,,,,,,,,,,
2013-10-09 14:06:00,529.666667,74.333333,0.0,68.666667,0.0,0.0,0.0,0.0,0.0,1.0,False,False,True
2013-10-09 14:07:00,543.875000,73.875000,0.0,68.750000,0.0,0.0,0.0,0.0,0.0,1.0,False,False,True
2013-10-09 14:08:00,578.000000,73.100000,0.0,41.900000,0.0,0.0,0.0,0.0,0.0,1.0,False,False,True
2013-10-09 14:09:00,638.300000,72.800000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,False,False,True
2013-10-09 14:10:00,800.200000,72.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,False,False,True


In [6]:
# Save the cleaned data and reports to disk.

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = CLEANED_DIR / "house1_cleaned_1min.csv"
raw_report_path = CLEANED_DIR / "house1_raw_quality_report.json"
cleaning_report_path = CLEANED_DIR / "house1_cleaning_report.json"

cleaned_house1.to_csv(cleaned_path, index_label="Time")
write_json(raw_quality, raw_report_path)
write_json(cleaning_report, cleaning_report_path)

print(f"Saved: {cleaned_path}")
print(f"Saved: {raw_report_path}")
print(f"Saved: {cleaning_report_path}")

Saved: D:\FLOCK_ENERGY_ASSIGNMENT\refit-appliance-disaggregation\data\cleaned\house1_cleaned_1min.csv
Saved: D:\FLOCK_ENERGY_ASSIGNMENT\refit-appliance-disaggregation\data\cleaned\house1_raw_quality_report.json
Saved: D:\FLOCK_ENERGY_ASSIGNMENT\refit-appliance-disaggregation\data\cleaned\house1_cleaning_report.json


In [7]:
# Plot before and after cleaning for house 1.

valid_times = raw_house1["Time"].dropna()
if valid_times.empty:
    raise ValueError("House 1 has no valid timestamps to plot.")

plot_start = valid_times.min()
plot_end = min(plot_start + pd.Timedelta(days=1), valid_times.max())
plot_channels = [
    "Aggregate",
    *[column for column in raw_house1.columns if column.startswith("Appliance")][:2],
]
plot_path = PLOTS_DIR / "house1_raw_vs_cleaned.png"

plot_before_after(
    raw_house1,
    cleaned_house1,
    plot_path,
    plot_start,
    plot_end,
    plot_channels,
)

print(f"Saved: {plot_path}")
display(plt.imread(plot_path))

Saved: D:\FLOCK_ENERGY_ASSIGNMENT\refit-appliance-disaggregation\plots\house1_raw_vs_cleaned.png


array([[[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        ...,
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        ...,
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        ...,
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       ...,

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        ...,
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        ...,
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        ...,
        [1., 1., 1., 1.],
        [1., 1.

In [ ]:
# Perform basic checks on the cleaned data.

assert isinstance(cleaned_house1.index, pd.DatetimeIndex)
assert cleaned_house1.index.is_monotonic_increasing
assert cleaned_house1.index.is_unique
assert "short_gap_filled" in cleaned_house1.columns
assert "long_gap_missing" in cleaned_house1.columns
assert "observed" in cleaned_house1.columns

print("Part 1 checks passed.")
print(f"Cleaned rows: {len(cleaned_house1):,}")
print(f"Short-gap rows filled: {int(cleaned_house1['short_gap_filled'].sum()):,}")
print(f"Long-gap rows flagged: {int(cleaned_house1['long_gap_missing'].sum()):,}")

Part 1 checks passed.
Cleaned rows: 920,031
Short-gap rows filled: 8,352
Long-gap rows flagged: 214,751


: 